# RNA Seq Pipeline
#### Author: Kylie Gregory 
#### Created On: 11/25/2026
#### Last Modified: 11/25/2026
#### Purpose: Create an RNA-seq pipeline to determine what gene expression changes are consistent across two independent datasets comparing dilated cardiomyopathy (DCM) and healthy (non-failing donor) groups. 

#### Files: 


In [24]:
#load in libraries

library(DESeq2)
library(tidyverse)
library(GEOquery)

Loading required package: S4Vectors



Loading required package: stats4




Attaching package: ‘S4Vectors’




The following object is masked from ‘package:utils’:

    findMatches




The following objects are masked from ‘package:base’:

    expand.grid, I, unname




Loading required package: IRanges



... (8 outputs left out)

The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowCumsums, rowDiffs, rowIQRDiffs, rowIQRs, rowLogSumExps,
    rowMadDiffs, rowMads, rowMaxs, rowMeans2, rowMedians, rowMins,
    rowOrderStats, rowProds, rowQuantiles, rowRanges, rowRanks,
    rowSdDiffs, rowSds, rowSums2, rowTabulates, rowVarDiffs, rowVars,
    rowWeightedMads, rowWeightedMeans, rowWeightedMedians,
    rowWeightedSds, rowWeig

The following object is masked from ‘package:Biobase’:

    rowMedians




── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     


── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ lubridate::%within%() masks IRanges::%within%()
✖ dplyr::collapse()     masks IRanges::collapse()
✖ dplyr::combine()      masks Biobase::combine(), BiocGenerics::combine()
✖ dplyr::count()        masks matrixStats::count()
✖ dplyr::desc()         masks IRanges::desc()
✖ tidyr::expand()       masks S4Vectors::expand()
✖ dplyr::filter()       masks stats::filter()
✖ dplyr::first()        masks S4Vectors::first()
✖ dplyr::lag()          masks stats::lag()
✖ ggplot2::Position()   masks BiocGenerics::Position(), base::Position()
✖ purrr::reduce()       masks GenomicRanges::reduce(), IRanges::reduce()
✖ dplyr::rename()       masks S4Vectors::rename()
✖ lubridate::second()   masks S4Vectors::second()
✖ lubridate::second<-() masks S4Vectors::second<-()
✖ dplyr::slice()        masks IRanges::slice()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [ ]:
#read in RNA-seq raw counts dataset


create_raw_counts_table <- function(raw_counts_url, GEO_number){
    #either GSE141910.tsv.gz or GSE116250.tsv.gz
    file_name <- paste(GEO_number, ".tsv.gz", sep="")
    download.file(raw_counts_url, file_name)
    count_table <- read.table(file_name, header=T,sep="\t")
    #make GeneID column the unique row identifiers 
    count_table <- column_to_rownames(count_table, var ="GeneID")

    return(count_table)
    
}

In [48]:

#load in column data to map each sample to a condition 

create_column_data <- function(GEO_number
                               
    #get the GEO series                          
    gse <- getGEO(GEO_number)
                               
    #get the phenotypic data about each sample                           
    sample_data <- pData(gse[[1]])
    
     #the condition (DCM vs non-failing) is in different columns for the two datasets we are analysing, so created an if/else block 
                               
    #this bloc filters to only DCM vs non-failing samples, selects only that column, changes the name of the column to condition, and edits the exact wording of non-failing and DCM to be consistent among the datasets to allow downstream code to be modular
                              
                 
    if (GEO_number =="GSE141910")
        {
        col_data <- sample_data %>% filter(characteristics_ch1 %in% c("etiology: Non-Failing Donor",
                                                                    "etiology: Dilated cardiomyopathy
(DCM)")) %>% select(characteristics_ch1) %>% rename(condition=characteristics_ch1)
        
        col_data$condition <- sub("etiology: ", "", col_data$condition)
            
    } else{
        col_data <-  col_data <- sample_data %>% filter(`disease:ch1` %in% c("non-failing", "dilated cardiomyopathy")) %>% select(`disease:ch1`) %>% rename(condition=`disease:ch1`)
        
        col_data$condition <- sub("non-failing", "Non-Failing Donor", col_data$condition)
        col_data$condition <- sub("dilated cardiomyopathy", "Dilated cardiomyopathy (DCM)", 
                                  col_data$condition)
    
    }
    
    
    return (col_data)

}







In [49]:
d1_geo_number <- "GSE141910"
d2_geo_number <- "GSE116250"

d1_count_url <- "https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc=GSE141910&format=file&file=GSE141910_raw_counts_GRCh38.p13_NCBI.tsv.gz"
d2_count_url <- "https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc=GSE116250&format=file&file=GSE116250_raw_counts_GRCh38.p13_NCBI.tsv.gz"


d1_count_table <- create_raw_counts_table(d1_count_url, d1_geo_number)
d2_count_table <- create_raw_counts_table(d2_count_url, d2_geo_number)


d1_col_data <- create_column_data(d1_geo_number)
d2_col_data <- create_column_data(d2_geo_number)
    

Found 1 file(s)



GSE116250_series_matrix.txt.gz



Using locally cached version: /tmp/Rtmpe1LBNQ/GSE116250_series_matrix.txt.gz



Using locally cached version of GPL16791 found here:
/tmp/Rtmpe1LBNQ/GPL16791.soft.gz 



,condition
,<chr>
GSM3219558,Non-Failing Donor
GSM3219559,Non-Failing Donor
GSM3219560,Non-Failing Donor
GSM3219561,Non-Failing Donor
GSM3219562,Non-Failing Donor
GSM3219563,Non-Failing Donor
GSM3219564,Non-Failing Donor
GSM3219565,Non-Failing Donor
GSM3219566,Non-Failing Donor
